In [5]:
import cv2
import numpy as np

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

lower_red1 = np.array([0, 120, 70])
upper_red1 = np.array([10, 255, 255])

lower_red2 = np.array([170, 120, 70])
upper_red2 = np.array([180, 255, 255])

canvas = None
prev_point = None

print("Air Canvas Started! Use a RED object. Press 'c' to clear and 'q' to quit.")

while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    if canvas is None:
        canvas = np.zeros_like(frame)

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = cv2.bitwise_or(mask1, mask2)

    mask = cv2.erode(mask, None, iterations=1)
    mask = cv2.dilate(mask, None, iterations=1)

    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    curr_point = None

    if contours:
        contour = max(contours, key=cv2.contourArea)

        if cv2.contourArea(contour) > 500:
            M = cv2.moments(contour)

            if M["m00"] != 0:
                curr_point = (
                    int(M["m10"] / M["m00"]),
                    int(M["m01"] / M["m00"])
                )

                cv2.circle(frame, curr_point, 8, (0, 255, 255), -1)

    if curr_point and prev_point:
        cv2.line(canvas, prev_point, curr_point, (0, 0, 255), 5)

    prev_point = curr_point

    combined = cv2.add(frame, canvas)

    cv2.putText(
        combined,
        "C: Clear | Q: Quit",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imshow("Air Canvas", combined)

    key = cv2.waitKey(1) & 0xFF

    if key == ord("c"):
        canvas = np.zeros_like(frame)
    elif key == ord("q"):
        break

cap.release()


Air Canvas Started! Use a RED object. Press 'c' to clear and 'q' to quit.
